In [2]:
pip install transformer torch

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement transformer (from versions: none)
ERROR: No matching distribution found for transformer


In [3]:
from transformers import pipeline
import torch
device=0 if torch.cuda.is_available() else -1

In [4]:
sentiment_model=pipeline("text-classification",
                         model="pysentimiento/bertweet-pt-sentiment",
                         device=device,
                         truncation=True,
                         batch_size=128,
                         max_length=128)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: pysentimiento/bertweet-pt-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
import pandas as pd
has_comment=pd.read_csv(r"D:\ai_customer_intelligence\data\master_table_dataset.csv")
has_comment=has_comment[has_comment['comment']!='sem comentários']
has_comment.shape

(45196, 13)

In [6]:
print(len(has_comment))

45196


In [7]:

label_map = {'POS': 'Positive', 'NEG': 'Negative', 'NEU': 'Neutral'}

def run_sentiment(text, batch_size=128):   # ✅ FIX 3: increased default batch_size
    scores = []
    labels = []

    for i in range(0, len(text), batch_size):
        batch = [str(t)[:512] for t in text[i:i + batch_size]]
        result = sentiment_model(batch)

        for r in result:
            raw_label = r['label']
            confidence = r['score']
            mapped = label_map.get(raw_label, 'Neutral')   # ✅ FIX 4: use label_map safely
            labels.append(mapped)

            if raw_label == 'POS':
                scores.append(confidence)
            elif raw_label == 'NEG':
                scores.append(-confidence)
            else:
                scores.append(0.0)

        # ✅ FIX 5: print every batch, not just multiples of 5000
        print(f"Processed: {min(i + batch_size, len(text))}/{len(text)}")

    return scores, labels

# ✅ FIX 6: safer column assignment
results = run_sentiment(has_comment['comment'].tolist())
has_comment['sentiment_score'] = results[0]
has_comment['sentiment'] = results[1]

Processed: 128/45196
Processed: 256/45196
Processed: 384/45196
Processed: 512/45196
Processed: 640/45196
Processed: 768/45196
Processed: 896/45196
Processed: 1024/45196
Processed: 1152/45196
Processed: 1280/45196
Processed: 1408/45196
Processed: 1536/45196
Processed: 1664/45196
Processed: 1792/45196
Processed: 1920/45196
Processed: 2048/45196
Processed: 2176/45196
Processed: 2304/45196
Processed: 2432/45196
Processed: 2560/45196
Processed: 2688/45196
Processed: 2816/45196
Processed: 2944/45196
Processed: 3072/45196
Processed: 3200/45196
Processed: 3328/45196
Processed: 3456/45196
Processed: 3584/45196
Processed: 3712/45196
Processed: 3840/45196
Processed: 3968/45196
Processed: 4096/45196
Processed: 4224/45196
Processed: 4352/45196
Processed: 4480/45196
Processed: 4608/45196
Processed: 4736/45196
Processed: 4864/45196
Processed: 4992/45196
Processed: 5120/45196
Processed: 5248/45196
Processed: 5376/45196
Processed: 5504/45196
Processed: 5632/45196
Processed: 5760/45196
Processed: 5888/4

In [11]:
has_comment=has_comment.reset_index(drop=True)

In [14]:
has_comment.shape

(45196, 15)

In [17]:
print(has_comment.columns)

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'product_id', 'price', 'product_category_name',
       'product_category_name_english', 'score', 'comment', 'month', 'year',
       'year_month', 'sentiment_score', 'sentiment'],
      dtype='object')


In [18]:
has_comment=has_comment[['order_id','score','comment','sentiment_score','sentiment']]

In [19]:
has_comment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45196 entries, 0 to 45195
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         45196 non-null  object 
 1   score            45196 non-null  float64
 2   comment          45196 non-null  object 
 3   sentiment_score  45196 non-null  float64
 4   sentiment        45196 non-null  object 
dtypes: float64(2), object(3)
memory usage: 1.7+ MB


In [20]:
has_comment.head(50)

,order_id,score,comment,sentiment_score,sentiment
0,e481f51cbdc54678b7cc49136f2d6af7,4.0,"não testei produto ainda, mas ele veio correto...",-0.822176,Negative
1,53cdb2fc8bc7dce0b6741e2150273451,4.0,muito bom produto.,0.992790,Positive
2,949d5b44dbf5de918fe9c16f97b45f8a,5.0,produto foi exatamente que eu esperava estava ...,0.000000,Neutral
3,e6ce16cb79ec1d90b1da9085a6118aeb,1.0,aguardando retorno da loja,0.000000,Neutral
4,e6ce16cb79ec1d90b1da9085a6118aeb,1.0,aguardando retorno da loja,0.000000,Neutral
5,432aaf21d85167c2c86ec9448c4e42cc,4.0,gostei do produto,0.990829,Positive
6,dcb36b511fcac050b97cd5c05de84dc3,5.0,obrigado pela atenção. lojas lannister perfeit...,0.953048,Positive
7,203096f03d82e0dffbc41ebc2e2bcfb7,2.0,os correios estäo em greve.. näo recebi nenhum...,-0.750174,Negative
8,f3e7c359154d965827355f39d6b1fdac,5.0,sempre vou comprar aqui pois melhor parabéns,0.992258,Positive
9,fbf9ac61453ac646ce8ad9783d7d0af6,2.0,demora muito entregar. já passou prazo ainda n...,-0.915909,Negative


In [21]:
has_comment.to_csv(r"D:\ai_customer_intelligence\data\reviews_with_sentiment_dataset.csv")